In [ ]:
import json
import ee
ee.Authenticate()
ee.Initialize(project="gen-lang-client-0412358476")
with open("../config/aoi.geojson") as f:
    aoi_points = json.load(f)

aois = {}
for nama, info in aoi_points.items():
    lon, lat = info["coord"]
    aois[nama] = ee.Geometry.Point([lon, lat]).buffer(info["buffer_m"])

print(f"Total {len(aois)} AOI dimuat")


In [ ]:
# --- Konstanta ---
CLOUD_FILTER   = 20     # filter metadata scene: buang scene >20% cloud cover
CLD_PRB_THRESH = 50     # s2cloudless: probability >50% dianggap awan
NIR_DRK_THRESH = 0.15   # B8 di bawah nilai ini (×10000) = kandidat dark pixel/shadow
CLD_PRJ_DIST   = 1      # jarak proyeksi shadow dari cloud (km)
BUFFER         = 50     # dilasi mask cloud (meter)

In [ ]:
AOI = aois['Titik_01']

In [ ]:
YEAR = 2024

In [ ]:
def add_mndwi(img):
    mndwi = img.normalizedDifference(['B3', 'B11']).rename('MNDWI')
    return img.addBands(mndwi)

def otsu(histogram):
    """Otsu threshold dari ee.Reducer.histogram — full server-side."""
    counts = ee.Array(ee.Dictionary(histogram).get('histogram'))
    means = ee.Array(ee.Dictionary(histogram).get('bucketMeans'))
    size = means.length().get([0])
    total = counts.reduce(ee.Reducer.sum(), [0]).get([0])
    sum_ = means.multiply(counts).reduce(ee.Reducer.sum(), [0]).get([0])
    mean = sum_.divide(total)

    indices = ee.List.sequence(1, size)

    def bss(i):
        a_counts = counts.slice(0, 0, i)
        a_count = a_counts.reduce(ee.Reducer.sum(), [0]).get([0])
        a_means = means.slice(0, 0, i)
        a_mean = a_means.multiply(a_counts).reduce(ee.Reducer.sum(), [0]).get([0]).divide(a_count)
        b_count = total.subtract(a_count)
        b_mean = sum_.subtract(a_count.multiply(a_mean)).divide(b_count)
        return a_count.multiply(a_mean.subtract(mean).pow(2)).add(
               b_count.multiply(b_mean.subtract(mean).pow(2)))

    bss_values = indices.map(lambda i: bss(ee.Number(i)))
    return means.sort(bss_values).get([-1])  # mean di index dgn BSS maksimum

In [4]:
def get_s2_sr_cld_col(aoi, start_date, end_date):
    s2_sr_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER)))
    s2_cloudless_col = (ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
        .filterBounds(aoi)
        .filterDate(start_date, end_date))
    return ee.ImageCollection(ee.Join.saveFirst('s2cloudless').apply(**{
        'primary': s2_sr_col,
        'secondary': s2_cloudless_col,
        'condition': ee.Filter.equals(**{
            'leftField': 'system:index', 'rightField': 'system:index'
        })
    }))

def add_cloud_bands(img):
    cld_prb = ee.Image(img.get('s2cloudless')).select('probability')
    is_cloud = cld_prb.gt(CLD_PRB_THRESH).rename('clouds')
    return img.addBands(ee.Image([cld_prb, is_cloud]))

def add_shadow_bands(img):
    not_water = img.select('SCL').neq(6)
    SR_BAND_SCALE = 1e4
    dark_pixels = img.select('B8').lt(NIR_DRK_THRESH * SR_BAND_SCALE).multiply(not_water).rename('dark_pixels')
    shadow_azimuth = ee.Number(90).subtract(ee.Number(img.get('MEAN_SOLAR_AZIMUTH_ANGLE')))
    cld_proj = (img.select('clouds').directionalDistanceTransform(shadow_azimuth, CLD_PRJ_DIST * 10)
        .reproject(**{'crs': img.select(0).projection(), 'scale': 100})
        .select('distance').mask().rename('cloud_transform'))
    shadows = cld_proj.multiply(dark_pixels).rename('shadows')
    return img.addBands(ee.Image([dark_pixels, cld_proj, shadows]))

def add_cld_shdw_mask(img):
    img_cloud = add_cloud_bands(img)
    img_cloud_shadow = add_shadow_bands(img_cloud)
    is_cld_shdw = img_cloud_shadow.select('clouds').add(img_cloud_shadow.select('shadows')).gt(0)
    is_cld_shdw = (is_cld_shdw.focalMin(2).focalMax(BUFFER * 2 / 20)
        .reproject(**{'crs': img.select([0]).projection(), 'scale': 20})
        .rename('cloudmask'))
    return img_cloud_shadow.addBands(is_cld_shdw)

def apply_cld_shdw_mask(img):
    not_cld_shdw = img.select('cloudmask').Not()
    return img.select('B.*').updateMask(not_cld_shdw)

In [ ]:
# --- Terapkan ke composite ---

coll = get_s2_sr_cld_col(AOI, f"{YEAR}-05-01", f"{YEAR}-10-31")
composite = (coll.map(add_cld_shdw_mask)
                 .map(apply_cld_shdw_mask)
                 .map(add_mndwi)
                 .median()
                 .clip(AOI))

# --- Hitung Otsu threshold server-side ---
histogram = composite.select('MNDWI').reduceRegion(
    reducer=ee.Reducer.histogram(255, 0.01),
    geometry=AOI,
    scale=10,
    maxPixels=1e9
).get('MNDWI')

threshold = otsu(histogram)
print('Otsu threshold:', threshold.getInfo())

# --- Buat binary water mask pakai threshold itu ---
water = composite.select('MNDWI').gt(threshold)
water_clean = (water
    .focalMode(radius=1, kernelType='square')  # buang salt-pepper noise
    .focalMax(1).focalMin(1))  

Otsu threshold: 0.08494333631318549


In [13]:
# ============================================================
# 02_PREPROCESSING — ORCHESTRATOR DUA JALUR
# Jalur A: composite → water mask bersih (input ConvLSTM)
# Jalur B: per-scene → mask + edge per scene + timestamp (bahan pyTMD/transect)
# ============================================================

# ---------- Sub-routine bersama ----------
def compute_water_mask(img, threshold):
    """MNDWI > threshold, lalu morphological cleanup (mode + closing)."""
    water = img.select('MNDWI').gt(threshold)
    return (water
            .focalMode(radius=1, kernelType='square')
            .focalMax(1).focalMin(1)
            .rename('water'))

def extract_edge(water):
    """Garis pantai 1-pixel: water dikurangi erosi-nya sendiri."""
    return water.subtract(water.focalMin(1)).selfMask().rename('shoreline')


# ---------- JALUR A: composite (ConvLSTM) ----------
def track_a_composite(coll, aoi):
    """Median composite musim -> MNDWI -> Otsu -> water_clean + edge.
    Return dict semua produk Jalur A."""
    composite = (coll.map(add_cld_shdw_mask)
                     .map(apply_cld_shdw_mask)
                     .map(add_mndwi)
                     .median()
                     .clip(aoi))

    hist = composite.select('MNDWI').reduceRegion(
        reducer=ee.Reducer.histogram(255, 0.01),
        geometry=aoi, scale=10, maxPixels=1e9).get('MNDWI')
    threshold = otsu(hist)

    water_clean = compute_water_mask(composite, threshold)
    edge = extract_edge(water_clean)

    return {'composite': composite, 'threshold': threshold,
            'water': water_clean, 'edge': edge}


# ---------- JALUR B: per-scene (transect/pyTMD) ----------
def track_b_per_scene(coll, aoi, threshold):
    """Proses TIAP scene individual: mask -> edge, timestamp dipertahankan.
    Threshold dari Jalur A dipakai konsisten. Return (ImageCollection, DataFrame timestamp)."""
    def per_scene(img):
        img_masked = apply_cld_shdw_mask(add_cld_shdw_mask(img))
        mndwi = img_masked.normalizedDifference(['B3', 'B11']).rename('MNDWI')
        water = compute_water_mask(mndwi.addBands(mndwi), threshold)
        edge = extract_edge(water)
        return (ee.Image.cat([water, edge])
                .clip(aoi)
                .copyProperties(img, ['system:time_start', 'system:index']))

    scenes = coll.map(per_scene)

    import pandas as pd
    ts = coll.aggregate_array('system:time_start').getInfo()
    ids = coll.aggregate_array('system:index').getInfo()
    df = pd.DataFrame({'scene_id': ids,
                       'timestamp': pd.to_datetime(ts, unit='ms')})
    return scenes, df


# ---------- ORCHESTRATOR ----------
def orchestrate_preprocessing(aoi, year, start_md="05-01", end_md="10-31"):
    """Entry point tunggal: satu collection sumber -> dua jalur output."""
    coll = get_s2_sr_cld_col(aoi, f"{year}-{start_md}", f"{year}-{end_md}")
    n = coll.size().getInfo()
    print(f"[orchestrator] {n} scene di collection {year}")

    # Jalur A dulu (threshold-nya dipakai Jalur B biar konsisten)
    a = track_a_composite(coll, aoi)
    t_val = a['threshold'].getInfo()
    print(f"[Jalur A] Otsu threshold = {t_val:.4f} | composite + water_clean siap")

    # Jalur B pakai threshold yang sama
    b_scenes, b_df = track_b_per_scene(coll, aoi, a['threshold'])
    print(f"[Jalur B] {len(b_df)} scene diproses per-scene, timestamp tersimpan")

    return {'track_a': a, 'track_b_scenes': b_scenes, 'track_b_timestamps': b_df}


# ---------- Jalankan ----------
result = orchestrate_preprocessing(AOI, 2024)
result['track_b_timestamps'].head(10)

[orchestrator] 31 scene di collection 2024
[Jalur A] Otsu threshold = 0.0849 | composite + water_clean siap
[Jalur B] 31 scene diproses per-scene, timestamp tersimpan


,scene_id,timestamp
0,20240502T023551_20240502T025256_T49MDP,2024-05-02 02:59:40.887
1,20240507T023529_20240507T025421_T49MDP,2024-05-07 02:59:38.804
2,20240515T024551_20240515T030911_T49MDP,2024-05-15 03:09:35.954
3,20240616T023529_20240616T025423_T49MDP,2024-06-16 02:59:40.928
4,20240621T023551_20240621T025648_T49MDP,2024-06-21 02:59:42.949
5,20240714T024531_20240714T030638_T49MDP,2024-07-14 03:09:35.309
6,20240716T023529_20240716T025424_T49MDP,2024-07-16 02:59:41.090
7,20240719T024529_20240719T030657_T49MDP,2024-07-19 03:09:36.580
8,20240724T024551_20240724T030651_T49MDP,2024-07-24 03:09:35.272
9,20240726T023529_20240726T025546_T49MDP,2024-07-26 02:59:40.701


In [17]:
import geemap
LONG = 110.4619860
LAT = -5.7972333

In [18]:
# ---------- Visualisasi preprocessing dua jalur ----------
a = result['track_a']
sample_scene = ee.Image(result['track_b_scenes'].first())

Map = geemap.Map(center=[LAT, LONG], zoom=15)
Map.add_basemap("SATELLITE")

# Jalur A
Map.addLayer(a['composite'], {'bands': ['B4','B3','B2'], 'min':0, 'max':2500, 'gamma':1.1},
             'A | RGB composite', True)
Map.addLayer(a['water'].selfMask(), {'palette': ['#2166ac']}, 'A | water_clean', False)
Map.addLayer(a['edge'], {'palette': ['#e31a1c']}, 'A | shoreline (composite)', True)

# Jalur B — contoh 1 scene individual buat pembanding
Map.addLayer(sample_scene.select('water').selfMask(), {'palette': ['#5aae61']},
             'B | water (1 scene)', False)
Map.addLayer(sample_scene.select('shoreline').selfMask(), {'palette': ['#ff7f00']},
             'B | shoreline (1 scene)', True)

Map.addLayer(AOI, {'color': 'yellow'}, 'AOI')
Map.addLayerControl()
Map

Map(center=[-5.7972333, 110.461986], controls=(WidgetControl(options=['position', 'transparent_bg'], position=…

In [19]:
# ============================================================
# 02_PREPROCESSING v2 — ORCHESTRATOR DUA JALUR + DIAGNOSTICS
# Jalur A: composite -> water mask bersih (input ConvLSTM)
# Jalur B: per-scene -> mask + edge per scene + timestamp (bahan pyTMD/transect)
#
# ASUMSI: sel sebelumnya di notebook sudah mendefinisikan:
#   - ee.Initialize()
#   - get_s2_sr_cld_col, add_cloud_bands, add_shadow_bands,
#     add_cld_shdw_mask, apply_cld_shdw_mask   (cloud/shadow masking)
#   - otsu()                                   (Otsu thresholding dari histogram)
#   - CLOUD_FILTER, CLD_PRB_THRESH, NIR_DRK_THRESH, BUFFER, CLD_PRJ_DIST
#   - AOI (ee.Geometry) sudah dimuat dari aoi.geojson
#
# Fungsi add_mndwi() lama TIDAK dipakai lagi -> diganti add_mndwi_resampled()
# di bawah, karena B3 (10m) vs B11 (20m) butuh alignment eksplisit.
# ============================================================

import ee
import pandas as pd


# ---------- Sub-routine bersama (DIPERBAIKI) ----------

def add_mndwi_resampled(img):
    """MNDWI dengan B11 di-resample BILINEAR ke grid 10m (native B3)
    sebelum dikombinasikan.

    Kenapa: normalizedDifference(['B3','B11']) langsung itu memaksa GEE
    align B3(10m) & B11(20m) pakai nearest-neighbor default -> transisi
    water/land jadi mengikuti kotak grid 20m persis (staircase kasar).
    Bilinear resample TIDAK menambah resolusi asli (informasi tetap dari
    20m), tapi menghaluskan transisi secara visual & mengurangi step besar.
    """
    b3 = img.select('B3')
    b11 = (img.select('B11')
              .resample('bilinear')
              .reproject(crs=b3.projection(), scale=10))
    mndwi = b3.subtract(b11).divide(b3.add(b11)).rename('MNDWI')
    return img.addBands(mndwi)


def compute_water_mask(mndwi_img, threshold, closing_radius=2, scale=10):
    """MNDWI > threshold -> morphological cleanup (mode denoise + closing).

    Perubahan dari v1:
    - closing_radius default naik dari 1 -> 2. Radius 1 cuma nutup gap
      1-pixel; gap dari shadow dermaga/kapal/turbidity biasanya lebih
      lebar dari itu.
    - units='pixels' di-set eksplisit di semua operasi focal supaya
      radius efektifnya konsisten & tidak ambigu antar scene.
    - reproject(scale=...) eksplisit di akhir supaya operasi focal jalan
      di grid yang sama persis dengan grid MNDWI (bukan default
      projection image yang bisa beda-beda).
    """
    water = mndwi_img.select('MNDWI').gt(threshold)
    proj = mndwi_img.select('MNDWI').projection()
    cleaned = (water
               .focalMode(radius=1, kernelType='square', units='pixels')
               .focalMax(closing_radius, kernelType='square', units='pixels')
               .focalMin(closing_radius, kernelType='square', units='pixels')
               .reproject(crs=proj, scale=scale)
               .rename('water'))
    return cleaned


def extract_edge(water):
    """Garis pantai 1-pixel: water dikurangi erosi-nya sendiri."""
    return water.subtract(water.focalMin(1, units='pixels')).selfMask().rename('shoreline')


def edge_to_simplified_vector(edge, aoi, scale=10, max_pixels=1e9, tolerance_m=15):
    """Konversi edge raster -> vector, lalu simplify.

    CATATAN JUJUR: GEE reduceToVectors tidak punya native LineString;
    hasilnya polygon tipis mengikuti bentuk edge (bukan polyline murni).
    simplify(tolerance_m) mengurangi jumlah vertex & menghaluskan sudut
    siku grid, tapi kalau lo butuh true polyline (misal utk baseline
    transect DSAS-style), biasanya perlu tahap tambahan di luar GEE:
    export ke GeoTIFF -> skeletonize (skimage) -> vectorize (rasterio/
    shapely) -> simplify. Fungsi ini cukup buat QA visual dulu.
    """
    vectors = edge.reduceToVectors(
        geometry=aoi, scale=scale, geometryType='polygon',
        maxPixels=max_pixels, eightConnected=True)
    simplified = vectors.map(lambda f: f.simplify(tolerance_m))
    return simplified


# ---------- JALUR A: composite (ConvLSTM) ----------

def track_a_composite(coll, aoi, closing_radius=2):
    """Median composite musim -> MNDWI (resampled) -> Otsu -> water_clean + edge."""
    composite = (coll.map(add_cld_shdw_mask)
                     .map(apply_cld_shdw_mask)
                     .map(add_mndwi_resampled)
                     .median()
                     .clip(aoi))

    hist = composite.select('MNDWI').reduceRegion(
        reducer=ee.Reducer.histogram(255, 0.01),
        geometry=aoi, scale=10, maxPixels=1e9).get('MNDWI')
    threshold = otsu(hist)

    water_clean = compute_water_mask(composite, threshold, closing_radius=closing_radius, scale=10)
    edge = extract_edge(water_clean)
    edge_vector = edge_to_simplified_vector(edge, aoi, scale=10)

    return {'composite': composite, 'threshold': threshold,
            'water': water_clean, 'edge': edge, 'edge_vector': edge_vector}


# ---------- JALUR B: per-scene (transect/pyTMD) ----------

def track_b_per_scene(coll, aoi, threshold, closing_radius=2):
    """Proses TIAP scene individual: mask -> edge, timestamp dipertahankan.
    Threshold dari Jalur A dipakai konsisten (divalidasi lewat diagnostic
    di bawah -- jangan diasumsikan valid tanpa dicek).

    Tambahan dari v1: valid_pct disimpan per scene (dipakai diagnostic
    cloud/shadow bias)."""
    def per_scene(img):
        img_masked = apply_cld_shdw_mask(add_cld_shdw_mask(img))
        img_mndwi = add_mndwi_resampled(img_masked)
        water = compute_water_mask(img_mndwi, threshold, closing_radius=closing_radius, scale=10)
        edge = extract_edge(water)

        valid_pct = (img_masked.select('B3').mask()
                     .reduceRegion(reducer=ee.Reducer.mean(), geometry=aoi,
                                   scale=10, maxPixels=1e9).get('B3'))

        return (ee.Image.cat([water, edge])
                .clip(aoi)
                .set('valid_pct', valid_pct)
                .copyProperties(img, ['system:time_start', 'system:index']))

    scenes = coll.map(per_scene)

    ts = coll.aggregate_array('system:time_start').getInfo()
    ids = coll.aggregate_array('system:index').getInfo()
    valid_pcts = scenes.aggregate_array('valid_pct').getInfo()

    df = pd.DataFrame({'scene_id': ids,
                       'timestamp': pd.to_datetime(ts, unit='ms'),
                       'valid_pct': valid_pcts})
    return scenes, df


# ---------- DIAGNOSTIC 1: threshold sharing sanity check ----------

def diagnostic_pct_water(scenes, aoi, scale=10):
    """% water pixel per scene pakai threshold global dari Jalur A.
    Dipakai buat lihat apakah threshold sharing (A -> B) itu valid,
    atau ada scene yang menyimpang jauh (sun glint, turbidity musiman, dll)."""
    def pct_water(img):
        stat = img.select('water').reduceRegion(
            reducer=ee.Reducer.mean(), geometry=aoi, scale=scale, maxPixels=1e9)
        return img.set('pct_water', stat.get('water'))
    checked = scenes.map(pct_water)
    return checked.aggregate_array('pct_water').getInfo()


def flag_outlier_scenes(df, col='pct_water', z_thresh=2.0):
    """Tandai scene yang pct_water-nya keluar dari range wajar (|z|>z_thresh).
    Scene yang ke-flag berarti threshold global kemungkinan salah klasifikasi
    di situ -> perlu threshold adaptif atau exclude dari Jalur B."""
    mean, std = df[col].mean(), df[col].std()
    df = df.copy()
    df['zscore'] = (df[col] - mean) / std if std > 0 else 0
    df['flag_outlier'] = df['zscore'].abs() > z_thresh
    return df


# ---------- EKSPERIMEN A: isolasi resolusi ----------

def experiment_resolution_isolation(scene_img, threshold, aoi):
    """Bandingkan edge RAW (nearest-neighbor default GEE, native 20m)
    vs edge RESAMPLED (bilinear ke 10m) dari SATU scene yang sama.

    Cara pakai: hasilnya 2 ee.Image, overlay keduanya di geemap.Map /
    folium punya lo. Kalau staircase jauh berkurang di versi resampled,
    itu bukti empiris bagian staircase itu artefak alignment -- bukan
    kesalahan segmentasi."""
    masked = apply_cld_shdw_mask(add_cld_shdw_mask(scene_img))

    # (a) RAW: normalizedDifference biasa, GEE align nearest-neighbor default
    mndwi_raw = masked.normalizedDifference(['B3', 'B11']).rename('MNDWI')
    water_raw = compute_water_mask(masked.addBands(mndwi_raw), threshold,
                                    closing_radius=1, scale=20)
    edge_raw = extract_edge(water_raw)

    # (b) RESAMPLED: bilinear ke grid 10m
    masked_r = add_mndwi_resampled(masked)
    water_resampled = compute_water_mask(masked_r, threshold,
                                          closing_radius=1, scale=10)
    edge_resampled = extract_edge(water_resampled)

    return {'edge_raw_20m': edge_raw.clip(aoi),
            'edge_resampled_10m': edge_resampled.clip(aoi)}


# ---------- EKSPERIMEN B: isolasi tide (same-month overlay) ----------

def experiment_tide_isolation(b_scenes, b_df, month, year):
    """Ambil semua scene DALAM BULAN & TAHUN yang sama (erosi/akresi riil
    diasumsikan ~nol dalam rentang sesingkat itu). Kalau posisi garis
    pantai masih geser signifikan antar scene ini, itu indikasi kuat
    tide effect -- bukan segmentation error atau erosi beneran.

    Cara pakai: overlay tiap scene hasil filter ini di geemap.Map,
    warnai per tanggal, lihat apakah jaraknya sebanding dengan yang lo
    lihat di screenshot AOI Titik_01."""
    mask = (b_df['timestamp'].dt.year == year) & (b_df['timestamp'].dt.month == month)
    ids_in_month = b_df.loc[mask, 'scene_id'].tolist()

    if len(ids_in_month) < 2:
        print(f"[warning] cuma {len(ids_in_month)} scene di {month}/{year}, "
              f"kurang buat overlay perbandingan.")
        return None

    filtered = b_scenes.filter(ee.Filter.inList('system:index', ids_in_month))
    print(f"[experiment_tide_isolation] {len(ids_in_month)} scene di {month}/{year}: "
          f"{ids_in_month}")
    return filtered


# ---------- SCAFFOLD pyTMD (belum dijalankan) ----------

def attach_tide_predictions(b_df, lon, lat):
    """
    TODO -- belum jalan, ini cuma struktur integrasi.

    from pyTMD.compute import tide_elevations
    tide_m = tide_elevations(
        x=lon, y=lat, delta_time=b_df['timestamp'].values,
        DIRECTORY=<path model tide lokal, mis. FES2022/EOT20>,
        MODEL='FES2022', EPSG=4326, TYPE='drift', TIME='datetime'
    )
    b_df['tide_height_m'] = tide_m

    Catatan:
    - Model tide (FES2022/EOT20) harus di-download manual, environment
      ini nggak ada akses ke server distribusinya.
    - tide_height_m masih VERTICAL. Buat convert ke horizontal shoreline
      shift butuh slope pantai lokal (dari DEM/survey):
          offset_m = tide_height_m / tan(slope)
      Baru offset_m ini yang dipakai koreksi posisi transect per scene.
    - Simpan tide_height_m & offset_m di b_df juga (bukan cuma di
      variable terpisah) supaya auditable per scene_id.
    """
    raise NotImplementedError("Isi path model tide dulu sebelum dipanggil.")


# ---------- ORCHESTRATOR ----------

def orchestrate_preprocessing(aoi, year, start_md="05-01", end_md="10-31",
                               closing_radius=2, run_diagnostics=True):
    """Entry point tunggal: satu collection sumber -> dua jalur output + diagnostics."""
    coll = get_s2_sr_cld_col(aoi, f"{year}-{start_md}", f"{year}-{end_md}")
    n = coll.size().getInfo()
    print(f"[orchestrator] {n} scene di collection {year}")

    a = track_a_composite(coll, aoi, closing_radius=closing_radius)
    t_val = a['threshold'].getInfo()
    print(f"[Jalur A] Otsu threshold = {t_val:.4f} | composite + water_clean siap")

    b_scenes, b_df = track_b_per_scene(coll, aoi, a['threshold'], closing_radius=closing_radius)
    print(f"[Jalur B] {len(b_df)} scene diproses per-scene, timestamp + valid_pct tersimpan")

    if run_diagnostics:
        pct_water = diagnostic_pct_water(b_scenes, aoi, scale=10)
        b_df['pct_water'] = pct_water
        b_df = flag_outlier_scenes(b_df, col='pct_water')
        n_flag = int(b_df['flag_outlier'].sum())
        print(f"[diagnostic 1] {n_flag} scene keluar dari range wajar pct_water "
              f"(|z|>2) -> threshold sharing patut dicurigai di scene ini")

        n_low_valid = int((b_df['valid_pct'] < 0.5).sum())
        print(f"[diagnostic 2] {n_low_valid} scene valid_pct < 50% "
              f"(cloud/shadow makan >separuh AOI) -> edge scene ini kurang reliable")

        month_counts = b_df['timestamp'].dt.month.value_counts().sort_index()
        print(f"[diagnostic 2b] distribusi scene per bulan:\n{month_counts}")

    return {'track_a': a, 'track_b_scenes': b_scenes, 'track_b_timestamps': b_df}


# ---------- Jalankan ----------
if __name__ == "__main__":
    result = orchestrate_preprocessing(AOI, 2024)
    print(result['track_b_timestamps'].head(10))

    # Eksperimen isolasi resolusi (1 scene contoh)
    coll_2024 = get_s2_sr_cld_col(AOI, "2024-05-01", "2024-10-31")
    one_scene = ee.Image(coll_2024.first())
    exp_a = experiment_resolution_isolation(one_scene, result['track_a']['threshold'], AOI)
    # -> exp_a['edge_raw_20m'] dan exp_a['edge_resampled_10m'], overlay di geemap.Map lo

    # Eksperimen isolasi tide (ganti month sesuai bulan yang scene-nya cukup banyak)
    exp_b = experiment_tide_isolation(result['track_b_scenes'],
                                       result['track_b_timestamps'],
                                       month=6, year=2024)

[orchestrator] 31 scene di collection 2024
[Jalur A] Otsu threshold = 0.0850 | composite + water_clean siap
[Jalur B] 31 scene diproses per-scene, timestamp + valid_pct tersimpan


ValueError: Length of values (30) does not match length of index (31)